In [721]:
import sys
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_extraction import DictVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    roc_auc_score,
    precision_score,
    recall_score,
    confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
import xgboost as xgb
import time
from pathlib import Path

In [722]:
THRESHOLD = 5
RANDOM_STATE = 30
FOLDER_OUTPUT = Path('data') / 'data_models'

In [723]:
requirments_name = []

with open("requirments.txt", "r", encoding="utf-8") as requirments:
    for r in requirments:
        text = r.replace("\n", "")
        requirments_name.append(text.split("=="))

for requirment_name in requirments_name:
    print(f"{requirment_name[0]} == {requirment_name[1]}")

version_system = sys.version.split()
print(f"version Python == {(version_system[0])}")


numpy == 2.5.4
pandas == 3.1.1
jupyter == 1.1.1
matplotlib == 3.11.2
ucimlrepo == 0.0.7
scikit-learn == 1.9.1
xgboost == 3.4.1
version Python == 3.12.10


In [724]:
data = []

with open('data/adult.data', 'r', encoding='utf-8') as file:
    for line in file:
      clear_data = line.replace("\n", "").split(", ")
      data.append(clear_data)


In [725]:
data_test = []

with open("data/adult.test", "r", encoding="utf-8") as file:
    for line in file:
        clear_data = line.replace("\n", "").replace('.', '').split(", ")
        data_test.append(clear_data)


In [726]:
columns = []

with open("data/adult.names", "r", encoding="utf-8") as file:
    lines = file.readlines()
    for line in range(96, 110):
      clear_data = lines[line].replace('\n', '').split(':')
      columns.append(clear_data[0])


columns.append('income')


In [727]:
df = pd.DataFrame(data, columns=columns)
df_test = pd.DataFrame(data_test, columns=columns)

C:\Users\TgSBot_xD\AppData\Local\Temp\ipykernel_20020\971865605.py:1: Pandas4Warning: Constructing a DataFrame from a list of sequences with mismatched lengths is deprecated and will raise in a future version. The shorter sequences are currently padded with NaN; make all sequences the same length before constructing instead.
  df = pd.DataFrame(data, columns=columns)
C:\Users\TgSBot_xD\AppData\Local\Temp\ipykernel_20020\971865605.py:2: Pandas4Warning: Constructing a DataFrame from a list of sequences with mismatched lengths is deprecated and will raise in a future version. The shorter sequences are currently padded with NaN; make all sequences the same length before constructing instead.
  df_test = pd.DataFrame(data_test, columns=columns)


In [728]:
index_null_value = df.index[df.isna().any(axis=1)]
index_null_value_test = df_test.index[df_test.isna().any(axis=1)]

if not len(index_null_value) == 0:
    print(f"Delete line from train data frame - {index_null_value.to_list()}")
    df.iloc[index_null_value]
    df.drop(index_null_value.to_list(), inplace=True)

else:
  print("doesn't have empty line for train data frame")


if not len(index_null_value_test) == 0:
    print(f"Delete line from train data frame - {index_null_value_test.to_list()}")
    df_test.iloc[index_null_value_test]
    df_test.drop(index_null_value_test.to_list(), inplace=True)

else:
    print("doesn't have empty line")

Delete line from train data frame - [32561]
Delete line from train data frame - [16281]


C:\Users\TgSBot_xD\AppData\Local\Temp\ipykernel_20020\1686725349.py:7: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(index_null_value.to_list(), inplace=True)
C:\Users\TgSBot_xD\AppData\Local\Temp\ipykernel_20020\1686725349.py:16: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_test.drop(index_null_value_test.to_list(), inplace=True)


In [729]:
df_test = df_test.replace('?', 'unknown')
df = df.replace("?", "unknown")

In [730]:
object_columns = [
    "workclass",
    "education",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country",
]

numerical_columns = list(set(columns) - set(object_columns))
numerical_columns.remove("fnlwgt")

target_index_columns = numerical_columns.index('income')

target_columns = numerical_columns[target_index_columns]

del numerical_columns[target_index_columns]

df[target_columns] = df[target_columns].map({"<=50K": 0, ">50K": 1})
df_test[target_columns] = df_test[target_columns].map({"<=50K": 0, ">50K": 1})

df[object_columns] = df[object_columns].astype('object')
df_test[object_columns] = df_test[object_columns].astype("object")

df[numerical_columns] = df[numerical_columns].astype(float)
df_test[numerical_columns] = df_test[numerical_columns].astype(float)

del df['fnlwgt']
del df_test["fnlwgt"]

In [731]:
df[object_columns] = df[object_columns].map(lambda x: x.lower().replace('-', '_'))
df_test[object_columns] = df_test[object_columns].map(
    lambda x: x.lower().replace("-", "_")
)

df.columns = df.columns.str.lower().str.replace("-", "_")
df_test.columns = df_test.columns.str.lower().str.replace("-", "_")

In [732]:
for index, text in enumerate(object_columns):
    object_columns[index] = text.replace("-", "_")

for index, text in enumerate(numerical_columns):
    numerical_columns[index] = text.replace("-", "_")

In [733]:
balanced_target_value_count = df['income'].value_counts().values

percent_balance_target_value = balanced_target_value_count[0] / balanced_target_value_count[1]

print(
    f"Percentage of balance of the target value in train data frame *(1/0) = {percent_balance_target_value}"
)

Percentage of balance of the target value in train data frame *(1/0) = 3.152659099604642


In [734]:
X_train, X_val, y_train, y_val = train_test_split(
    df, df["income"], test_size=0.2, random_state=RANDOM_STATE, stratify=df["income"]
)

df_shuffled = shuffle(df_test, random_state=RANDOM_STATE)

X_test = df_shuffled

In [735]:
y_train = X_train['income']
y_val = X_val['income']
y_test = X_test["income"]

In [736]:
del X_train['income']
del X_val["income"]
del X_test["income"]

In [737]:
X_train.reset_index(drop=True, inplace=True)
X_val.reset_index(drop=True, inplace=True)
X_test.reset_index(drop=True, inplace=True)

y_train.reset_index(drop=True, inplace=True)
y_val.reset_index(drop=True, inplace=True)
y_test.reset_index(drop=True, inplace=True)

C:\Users\TgSBot_xD\AppData\Local\Temp\ipykernel_20020\963717809.py:1: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  X_train.reset_index(drop=True, inplace=True)
C:\Users\TgSBot_xD\AppData\Local\Temp\ipykernel_20020\963717809.py:2: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  X_val.reset_index(drop=True, inplace=True)
C:\Users\TgSBot_xD\AppData\Local\Temp\ipykernel_20020\963717809.py:3: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  X_test.reset_index(drop=Tru

In [738]:
dicts_train = X_train.to_dict(orient='records')
dicts_validation = X_val.to_dict(orient="records")
dicts_test = X_test.to_dict(orient="records")

dv = DictVectorizer(sparse=False)

X_train_encode = dv.fit_transform(dicts_train)
X_val_encoded = dv.transform(dicts_validation)

features_name = dv.get_feature_names_out()
numerical_mask = np.isin(features_name, numerical_columns)

In [739]:
scaler = StandardScaler()

X_train_scaled = X_train_encode.copy()
X_val_scaled = X_val_encoded.copy()
X_train_scaled[:, numerical_mask] = scaler.fit_transform(X_train_encode[:, numerical_mask])
X_val_scaled[:, numerical_mask] = scaler.transform(X_val_encoded[:, numerical_mask])

print(
    "Train scalaer date:",
    X_train_scaled.shape[0],
    "Validation scalaer date:",
    X_val_scaled.shape[0],
)
print(
    f"y_train scalaer date: {(y_train == 1).mean()}",
    f"y_val scalaer date: {(y_val == 1).mean()}",
)

Train scalaer date: 26048 Validation scalaer date: 6513
y_train scalaer date: 0.24082463144963145 y_val scalaer date: 0.24074927068939045


In [740]:
start_time_train_baseline_model = time.perf_counter()
baseline_model = float(y_train.mean())
stop_time_train_baseline_model = time.perf_counter()

time_train_baseline_model = stop_time_train_baseline_model - start_time_train_baseline_model

In [741]:
logistic_model_params = {
    "random_state": RANDOM_STATE,
    "solver": "lbfgs",
    "dual": False,
    "fit_intercept": True,
    "intercept_scaling": 1,
    "C": 0.3,
    "l1_ratio": 0,
    "class_weight": None,
    "tol": 1e-4,
    "max_iter": 1000,
}
logistic_model = LogisticRegression(**logistic_model_params)

start_time_train_logistic_model = time.perf_counter()
logistic_model.fit(X_train_scaled, y_train)
stop_time_train_logistic_model = time.perf_counter()

time_train_logistic_model = (
    stop_time_train_logistic_model - start_time_train_logistic_model
)
y_pred = logistic_model.predict_proba(X_val_scaled)[:, 1]
roc_auc = roc_auc_score(y_val, y_pred)

In [742]:
decision_tree_model_params = {"random_state": RANDOM_STATE, 'max_depth': 8, 'min_samples_leaf': 5, 'min_samples_split': 2, 'criterion': 'gini'}

decision_tree_model = DecisionTreeClassifier(**decision_tree_model_params)

start_time_train_decision_tree_model = time.perf_counter()
decision_tree_model.fit(X_train_encode, y_train)
stop_time_train_decision_tree_model = time.perf_counter()

time_train_decision_tree_model = (
    stop_time_train_decision_tree_model - start_time_train_decision_tree_model
)

In [743]:
random_forest_model_params = {"random_state": RANDOM_STATE, 'n_estimators': 75, 'max_depth': 18, 'min_samples_leaf': 3, 'max_features': 0.15}

random_forest_model = RandomForestClassifier(**random_forest_model_params)

start_time_train_random_forest_model = time.perf_counter()
random_forest_model.fit(X_train_encode, y_train)
stop_time_train_random_forest_model = time.perf_counter()

time_train_random_forest_model = (
    stop_time_train_random_forest_model - start_time_train_random_forest_model
)

In [744]:
xgboost_sklearn_model_params = {"random_state": RANDOM_STATE, 'learning_rate': 0.1, 'max_iter': 100, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'l2_regularization': 0, 'early_stopping': False}

xgboost_sklearn_model = HistGradientBoostingClassifier(**xgboost_sklearn_model_params)

start_time_train_xgboost_sklearn_model = time.perf_counter()
xgboost_sklearn_model.fit(X_train_encode, y_train)
stop_time_train_xgboost_sklearn_model = time.perf_counter()

time_train_xgboost_sklearn_model = (
    stop_time_train_xgboost_sklearn_model - start_time_train_xgboost_sklearn_model
)

In [745]:
features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train_encode, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val_encoded, label=y_val, feature_names=features)

xgboost_model_params = {
    "eta": 0.1,
    "max_depth": 8,
    "min_child_weight": 3,
    "objective": "binary:logistic",
    "nthread": 6,
    "seed": 1,
    "verbosity": 1,
    "tree_method": "hist",
    "eval_metric": ["auc", "logloss"],
    "subsample": 0.8,
    'colsample_bytree': 1,
    'lambda': 1,
    'gamma': 1,
    'alpha': 0
}

xgboost_start_params = {"num_boost_round": 100 }

xgboost_all_params = xgboost_start_params | xgboost_model_params

start_time_train_xgboost_model = time.perf_counter()
xgboost_model = xgb.train(xgboost_model_params, dtrain, **xgboost_start_params)
stop_time_train_xgboost_model = time.perf_counter()

time_train_xgboost_model = (
    stop_time_train_xgboost_model - start_time_train_xgboost_model
)

In [746]:
# baseline - roc_auc score
baseline_model_train_probabilities = np.full(len(y_train), baseline_model)
baseline_model_train_decisions = baseline_model_train_probabilities >= THRESHOLD

baseline_model_val_probabilities = np.full(len(y_val), baseline_model)
baseline_model_val_decisions = baseline_model_val_probabilities >= THRESHOLD

baseline_train_auc_score = 0.5
baseline_val_auc_score = 0.5

# logistic Regression - roc_auc score
logistic_model_train_roc_auc_score = roc_auc_score(
    y_train, logistic_model.predict_proba(X_train_scaled)[:, 1]
)
logistic_model_val_roc_auc_score = roc_auc_score(
    y_val, logistic_model.predict_proba(X_val_scaled)[:, 1]
)

# decision tree - roc_auc score
decision_tree_model_train_roc_auc_score = roc_auc_score(
    y_train, decision_tree_model.predict_proba(X_train_encode)[:, 1]
)
decision_tree_model_val_roc_auc_score = roc_auc_score(
    y_val, decision_tree_model.predict_proba(X_val_encoded)[:, 1]
)

# random_forest - roc_auc score
random_forest_model_train_roc_auc_score = roc_auc_score(
    y_train, random_forest_model.predict_proba(X_train_encode)[:, 1]
)
random_forest_model_val_roc_auc_score = roc_auc_score(
    y_val, random_forest_model.predict_proba(X_val_encoded)[:, 1]
)

# xgboost_sklearn - roc_auc score
xgboost_sklearn_model_train_roc_auc_score = roc_auc_score(
    y_train, xgboost_sklearn_model.predict_proba(X_train_encode)[:, 1]
)
xgboost_sklearn_model_val_roc_auc_score = roc_auc_score(
    y_val, xgboost_sklearn_model.predict_proba(X_val_encoded)[:, 1]
)

# xgboost - roc_auc score
xgboost_model_train_roc_auc_score = roc_auc_score(
    y_train, xgboost_model.predict(dtrain)
)
xgboost_model_val_roc_auc_score = roc_auc_score(y_val, xgboost_model.predict(dval))

In [747]:
#Log_loss - baseline
baseline_model_train_log_loss = log_loss(y_train, baseline_model_train_probabilities)
baseline_model_val_log_loss = log_loss(y_val, np.full(len(y_val), baseline_model))

# Log_loss - logistic
logistic_model_train_log_loss = log_loss(
    y_train, logistic_model.predict_proba(X_train_scaled)[:, 1]
)
logistic_model_val_log_loss = log_loss(
    y_val, logistic_model.predict_proba(X_val_scaled)[:, 1]
)

# Log_loss - decision_tree
decision_tree_model_train_log_loss = log_loss(
    y_train, decision_tree_model.predict_proba(X_train_encode)[:, 1]
)
decision_tree_model_val_log_loss = log_loss(
    y_val, decision_tree_model.predict_proba(X_val_encoded)[:, 1]
)

# Log_loss - random_forest
random_forest_model_train_log_loss = log_loss(
    y_train, random_forest_model.predict_proba(X_train_encode)[:, 1]
)
random_forest_model_val_log_loss = log_loss(
    y_val, random_forest_model.predict_proba(X_val_encoded)[:, 1]
)

# Log_loss - xgboost_sklearn
xgboost_sklearn_model_train_log_loss = log_loss(
    y_train, xgboost_sklearn_model.predict_proba(X_train_encode)[:, 1]
)
xgboost_sklearn_model_val_log_loss = log_loss(
    y_val, xgboost_sklearn_model.predict_proba(X_val_encoded)[:, 1]
)

# Log_loss - xgboost
xgboost_model_train_log_loss = log_loss(y_train, xgboost_model.predict(dtrain))
xgboost_model_val_log_loss = log_loss(y_val, xgboost_model.predict(dval))

In [748]:
# precision - baseline
baseline_model_train_probabilities = np.full(len(y_val), baseline_model)
baseline_model_probabilities = (baseline_model_train_probabilities <= 0.5).astype(int)
baseline_model_precision = precision_score(y_val, baseline_model_probabilities)

# precision - logistic
logistic_model_predict = logistic_model.predict_proba(X_val_encoded)[:, 1]
logistic_model_decision = (logistic_model_predict >= 0.5).astype(int)
logistic_model_precision = precision_score(
    y_val, logistic_model_decision
)

# precision - dicision tree
decision_tree_model_predict = decision_tree_model.predict_proba(X_val_encoded)[:, 1]
decision_tree_decision = (decision_tree_model_predict >= 0.5).astype(int)
decision_tree_precision = precision_score(y_val, decision_tree_decision)

# precision - random forest
random_forest_model_predict = random_forest_model.predict_proba(X_val_encoded)[:, 1]
random_forest_decision = (random_forest_model_predict >= 0.5).astype(int)
random_forest_model_precision = precision_score(y_val, random_forest_decision)

# precision - xgboost_sklearn
xgboost_sklearn_model_predict = xgboost_sklearn_model.predict_proba(X_val_encoded)[:, 1]
xgboost_sklearn_decision = (xgboost_sklearn_model_predict >= 0.5).astype(int)
xgboost_sklearnt_model_precision = precision_score(y_val, xgboost_sklearn_decision)

# precision - xgboost
xgboost_model_predict = xgboost_model.predict(dval)
xgboost_decision = (xgboost_model_predict >= 0.5).astype(int)
xgboost_model_precision = precision_score(y_val, xgboost_decision)

In [749]:
# recall - baseline
baseline_model_train_probabilities = np.full(len(y_val), baseline_model)
baseline_model_probabilities = (baseline_model_train_probabilities <= 0.5).astype(int)
baseline_model_recall = recall_score(y_val, baseline_model_probabilities)

# recall - logistic
logistic_model_predict = logistic_model.predict_proba(X_val_encoded)[:, 1]
logistic_model_decision = (logistic_model_predict >= 0.5).astype(int)
logistic_model_recall = recall_score(y_val, logistic_model_decision)

# recall - dicision tree
decision_tree_model_predict = decision_tree_model.predict_proba(X_val_encoded)[:, 1]
decision_tree_decision = (decision_tree_model_predict >= 0.5).astype(int)
decision_tree_recall = recall_score(y_val, decision_tree_decision)

# recall - random forest
random_forest_model_predict = random_forest_model.predict_proba(X_val_encoded)[:, 1]
random_forest_decision = (random_forest_model_predict >= 0.5).astype(int)
random_forest_model_recall = recall_score(y_val, random_forest_decision)

# recall - xgboost_sklearn
xgboost_sklearn_model_predict = xgboost_sklearn_model.predict_proba(X_val_encoded)[:, 1]
xgboost_sklearn_decision = (xgboost_sklearn_model_predict >= 0.5).astype(int)
xgboost_sklearnt_model_recall = recall_score(y_val, xgboost_sklearn_decision)

# recall - xgboost
xgboost_model_predict = xgboost_model.predict(dval)
xgboost_model_decision = (xgboost_model_predict >= 0.5).astype(int)
xgboost_model_recall = recall_score(y_val, xgboost_decision)


In [750]:
baseline_model_confusion_matrix = confusion_matrix(y_val, baseline_model_val_decisions)
logistic_model_confusion_matrix = confusion_matrix(
    y_val, logistic_model_decision
)
decision_model_confusion_matrix = confusion_matrix(y_val, decision_tree_decision)
random_forest_model_confusion_matrix = confusion_matrix(y_val, random_forest_decision)
xgboost_sklearn_model_confusion_matrix = confusion_matrix(y_val, xgboost_sklearn_decision)
xgboost_model_confusion_matrix = confusion_matrix(y_val, xgboost_model_decision)

In [751]:
# data models for a date frame

data_models = [
    {
        "method": "Baseline",
        "params": baseline_model,
        "train_roc_auc": baseline_train_auc_score,
        "val_roc_auc": baseline_val_auc_score,
        "train_log_loss": baseline_model_train_log_loss,
        "val_log_loss": baseline_model_val_log_loss,
        "recall_score": baseline_model_recall,
        "precision_score": baseline_model_precision,
        "confusion_matrix": baseline_model_confusion_matrix,
        "time_train": time_train_baseline_model,
    },
    {
        "method": "LogisticRegression",
        "params": logistic_model_params,
        "train_roc_auc": logistic_model_train_roc_auc_score,
        "val_roc_auc": logistic_model_val_roc_auc_score,
        "train_log_loss": logistic_model_train_log_loss,
        "val_log_loss": logistic_model_val_log_loss,
        "recall_score": logistic_model_recall,
        "precision_score": logistic_model_precision,
        "confusion_matrix": logistic_model_confusion_matrix,
        "time_train": time_train_logistic_model,
    },
    {
        "method": "Decision_Tree",
        "params": decision_tree_model_params,
        "train_roc_auc": decision_tree_model_train_roc_auc_score,
        "val_roc_auc": decision_tree_model_val_roc_auc_score,
        "train_log_loss": decision_tree_model_train_log_loss,
        "val_log_loss": decision_tree_model_val_log_loss,
        "recall_score": decision_tree_recall,
        "precision_score": decision_tree_precision,
        "confusion_matrix": decision_model_confusion_matrix,
        "time_train": time_train_decision_tree_model,
    },
    {
        "method": "Random_Forest",
        "params": random_forest_model_params,
        "train_roc_auc": random_forest_model_train_roc_auc_score,
        "val_roc_auc": random_forest_model_val_roc_auc_score,
        "train_log_loss": random_forest_model_train_log_loss,
        "val_log_loss": random_forest_model_val_log_loss,
        "recall_score": random_forest_model_recall,
        "precision_score": random_forest_model_precision,
        "confusion_matrix": random_forest_model_confusion_matrix,
        "time_train": time_train_random_forest_model,
    },
    {
        "method": "XGBoost_sklearn",
        "params": xgboost_sklearn_model_params,
        "train_roc_auc": xgboost_sklearn_model_train_roc_auc_score,
        "val_roc_auc": xgboost_sklearn_model_val_roc_auc_score,
        "train_log_loss": xgboost_sklearn_model_train_log_loss,
        "val_log_loss": xgboost_sklearn_model_val_log_loss,
        "recall_score": xgboost_sklearnt_model_recall,
        "precision_score": xgboost_sklearnt_model_precision,
        "confusion_matrix": xgboost_sklearn_model_confusion_matrix,
        "time_train": time_train_xgboost_sklearn_model,
    },
    {
        "method": "XGBoost",
        "params": xgboost_all_params,
        "train_roc_auc": xgboost_model_train_roc_auc_score,
        "val_roc_auc": xgboost_model_val_roc_auc_score,
        "train_log_loss": xgboost_model_train_log_loss,
        "val_log_loss": xgboost_model_val_log_loss,
        "recall_score": xgboost_model_recall,
        "precision_score": xgboost_model_precision,
        "confusion_matrix": xgboost_model_confusion_matrix,
        "time_train": time_train_xgboost_model,
    },
]

df_statistic_models = pd.DataFrame(data_models)

In [752]:
FOLDER_OUTPUT.mkdir(exist_ok=True)

for file_number in range(101):
    path = FOLDER_OUTPUT / f"try_{file_number}.csv"

    try:
        df_statistic_models.to_csv(
            path, mode="x", sep=";", encoding="utf-8", index=False
        )
    except FileExistsError:
        continue
    break


In [753]:
df_statistic_models

,method,params,train_roc_auc,val_roc_auc,train_log_loss,val_log_loss,recall_score,precision_score,confusion_matrix,time_train
0,Baseline,0.240825,0.500000,0.500000,0.552029,0.551942,1.000000,0.240749,"[[4945, 0], [1568, 0]]",0.000153
1,LogisticRegression,"{'random_state': 30, 'solver': 'lbfgs', 'dual'...",0.908207,0.907617,0.316699,0.319017,1.000000,0.240749,"[[0, 4945], [0, 1568]]",0.150113
2,Decision_Tree,"{'random_state': 30, 'max_depth': 8, 'min_samp...",0.913612,0.909857,0.296902,0.315484,0.547832,0.793900,"[[4722, 223], [709, 859]]",0.129814
3,Random_Forest,"{'random_state': 30, 'n_estimators': 75, 'max_...",0.946513,0.921525,0.258488,0.295463,0.588010,0.787361,"[[4696, 249], [646, 922]]",1.744051
4,XGBoost_sklearn,"{'random_state': 30, 'learning_rate': 0.1, 'ma...",0.939100,0.931517,0.260514,0.274435,0.651786,0.786759,"[[4668, 277], [546, 1022]]",0.565837
5,XGBoost,"{'num_boost_round': 100, 'eta': 0.1, 'max_dept...",0.945399,0.930719,0.248363,0.274643,0.659439,0.784522,"[[4661, 284], [534, 1034]]",0.315317


In [754]:
X_full_train = pd.concat([X_train, X_val], ignore_index=True)
y_full_train = pd.concat([y_train, y_val], ignore_index=True)

X_full_train_dict = X_full_train.to_dict(orient="records")


X_full_train_encode = dv.fit_transform(X_full_train_dict)
X_test_encoded = dv.transform(dicts_test)

# finish model
choose_finish_model = HistGradientBoostingClassifier(**xgboost_sklearn_model_params)

start_time_train_choose_model = time.perf_counter()
choose_finish_model.fit(X_full_train_encode, y_full_train)
stop_time_train_choose_model = time.perf_counter()

choose_model_prediction = choose_finish_model.predict_proba(X_test_encoded)[:, 1]
choose_model_decision = (choose_model_prediction >= THRESHOLD).astype(int)

time_train_choose_model = stop_time_train_choose_model - start_time_train_choose_model

In [755]:
# xgboost_sklearn - roc_auc score
choose_model_roc_auc_score = roc_auc_score(y_test, choose_model_prediction)

# Log_loss - xgboost_sklearn
choose_model_model_log_loss = log_loss(y_test, choose_model_prediction)

# Precision - xgboost_sklearn
choose_model_precision = precision_score(y_test, choose_model_decision, pos_label=1)

# Recall - xgboost_sklearn
choose_model_recall = recall_score(y_test, choose_model_decision, pos_label=1)

choose_model_confusion_matrix = confusion_matrix(y_test, choose_model_decision)

d:\UsefulPrograms\adult_income_classification\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [756]:
choose_finish_model_data = [
    {
        "method": "XGBoost_sklearn_finish_model",
        "params": xgboost_all_params,
        "roc_auc": choose_model_roc_auc_score,
        "log_loss": choose_model_model_log_loss,
        "recall_score": choose_model_recall,
        "precision_score": choose_model_precision,
        "confusion_matrix": choose_model_confusion_matrix,
        "time_train": time_train_choose_model,
    }
]

In [757]:
df_finish_model = pd.DataFrame(choose_finish_model_data)

In [758]:
df_finish_model


,method,params,roc_auc,log_loss,recall_score,precision_score,confusion_matrix,time_train
0,XGBoost_sklearn_finish_model,"{'num_boost_round': 100, 'eta': 0.1, 'max_dept...",0.927583,0.27695,0.0,0.0,"[[12435, 0], [3846, 0]]",0.560925
